# Phase 1 — Understand the Dataset & Business Problem

**Objective:** Identify the key drivers of employee attrition to help HR prioritize retention strategies.

**Target audience:** HR leadership and people analytics team.

**Key question:** Which employee segments and workplace factors have the strongest association with voluntary turnover?

### DLoad & Combine

In [1]:
import pandas as pd

In [2]:

train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")

In [3]:
print(train.shape)
print(test.shape)

(59598, 24)
(14900, 24)


In [4]:
df = pd.concat([train, test], ignore_index=True)

### Preprocessing & Cleaning

In [5]:
df.shape

(74498, 24)

In [6]:
df['Attrition'].value_counts()

,count
Attrition,
Stayed,39128
Left,35370


In [7]:
df['Attrition'].value_counts(normalize=True) * 100

,proportion
Attrition,
Stayed,52.522215
Left,47.477785


In [8]:
df.columns

Index(['Employee ID', 'Age', 'Gender', 'Years at Company', 'Job Role',
       'Monthly Income', 'Work-Life Balance', 'Job Satisfaction',
       'Performance Rating', 'Number of Promotions', 'Overtime',
       'Distance from Home', 'Education Level', 'Marital Status',
       'Number of Dependents', 'Job Level', 'Company Size', 'Company Tenure',
       'Remote Work', 'Leadership Opportunities', 'Innovation Opportunities',
       'Company Reputation', 'Employee Recognition', 'Attrition'],
      dtype='object')

# Data Audit

**What we check and why:**
- **Nulls** — missing data can bias group comparisons
- **Duplicates** — inflated counts distort rates
- **Dtypes** — ordinal columns need ordered categories for correct chart sorting
- **Class balance**

In [10]:
df.isnull().sum()

,0
Employee ID,0
Age,0
Gender,0
Years at Company,0
Job Role,0
Monthly Income,0
Work-Life Balance,0
Job Satisfaction,0
Performance Rating,0
Number of Promotions,0


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74498 entries, 0 to 74497
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Employee ID               74498 non-null  int64 
 1   Age                       74498 non-null  int64 
 2   Gender                    74498 non-null  object
 3   Years at Company          74498 non-null  int64 
 4   Job Role                  74498 non-null  object
 5   Monthly Income            74498 non-null  int64 
 6   Work-Life Balance         74498 non-null  object
 7   Job Satisfaction          74498 non-null  object
 8   Performance Rating        74498 non-null  object
 9   Number of Promotions      74498 non-null  int64 
 10  Overtime                  74498 non-null  object
 11  Distance from Home        74498 non-null  int64 
 12  Education Level           74498 non-null  object
 13  Marital Status            74498 non-null  object
 14  Number of Dependents  

In [12]:
df.duplicated().sum()

np.int64(0)

#### converting ordinal columns to ordered categories for cleaner visualizations.

In [13]:
cols = [
    "Work-Life Balance",
    "Job Satisfaction",
    "Performance Rating",
    "Education Level",
    "Job Level",
    "Company Size",
    "Company Reputation",
    "Employee Recognition"
]

for col in cols:
    print(f"\n{col}")
    print(sorted(df[col].unique()))


Work-Life Balance
['Excellent', 'Fair', 'Good', 'Poor']

Job Satisfaction
['High', 'Low', 'Medium', 'Very High']

Performance Rating
['Average', 'Below Average', 'High', 'Low']

Education Level
['Associate Degree', 'Bachelor’s Degree', 'High School', 'Master’s Degree', 'PhD']

Job Level
['Entry', 'Mid', 'Senior']

Company Size
['Large', 'Medium', 'Small']

Company Reputation
['Excellent', 'Fair', 'Good', 'Poor']

Employee Recognition
['High', 'Low', 'Medium', 'Very High']


In [14]:
import pandas as pd

# Work-Life Balance
df["Work-Life Balance"] = pd.Categorical(
    df["Work-Life Balance"],
    categories=["Poor", "Fair", "Good", "Excellent"],
    ordered=True
)

# Job Satisfaction
df["Job Satisfaction"] = pd.Categorical(
    df["Job Satisfaction"],
    categories=["Low", "Medium", "High", "Very High"],
    ordered=True
)

# Performance Rating
df["Performance Rating"] = pd.Categorical(
    df["Performance Rating"],
    categories=["Low", "Below Average", "Average", "High"],
    ordered=True
)

# Education Level
df["Education Level"] = pd.Categorical(
    df["Education Level"],
    categories=[
        "High School",
        "Associate Degree",
        "Bachelor’s Degree",
        "Master’s Degree",
        "PhD"
    ],
    ordered=True
)

# Job Level
df["Job Level"] = pd.Categorical(
    df["Job Level"],
    categories=[
        "Entry",
        "Mid",
        "Senior"
    ],
    ordered=True
)

# Company Size
df["Company Size"] = pd.Categorical(
    df["Company Size"],
    categories=[
        "Small",
        "Medium",
        "Large"
    ],
    ordered=True
)

# Company Reputation
df["Company Reputation"] = pd.Categorical(
    df["Company Reputation"],
    categories=[
        "Poor",
        "Fair",
        "Good",
        "Excellent"
    ],
    ordered=True
)

# Employee Recognition
df["Employee Recognition"] = pd.Categorical(
    df["Employee Recognition"],
    categories=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ],
    ordered=True
)

The dataset is relatively balanced — 52.7% stayed vs 47.3% left. This means we can compare rates directly without resampling.

In [15]:
import plotly.express as px

fig = px.pie(
    df,
    names="Attrition",
    title="Employee Attrition Distribution",
    hole=0.4
)

fig.show()

#### Which employee demographics are associated with attrition?

In [16]:
df.groupby("Attrition")["Age"].describe()

,count,mean,std,min,25%,50%,75%,max
Attrition,,,,,,,,
Left,35370.0,37.884111,12.245562,18.0,27.0,38.0,48.0,59.0
Stayed,39128.0,39.113371,11.905088,18.0,29.0,39.0,49.0,59.0


##### Insight
Employees who left average ~1 year younger than those who stayed. The gap is small and unlikely to be a primary driver — age alone should not inform retention strategy.

##### Insight: Employees who left tend to be slightly younger on average than those who stayed, although the difference is relatively modest.

###### Gender vs Attrition

In [17]:
pd.crosstab(
    df["Gender"],
    df["Attrition"],
    normalize="index"
)*100

Attrition,Left,Stayed
Gender,,
Female,53.011404,46.988596
Male,42.913829,57.086171


### insights and cta:One notable pattern is the difference in attrition rates between genders. Female employees showed a higher turnover rate than male employees, suggesting that additional analysis may be needed to identify workplace factors influencing retention within this group.

##### Marital Status vs Attrition

In [18]:
pd.crosstab(
    df["Marital Status"],
    df["Attrition"],
    normalize="index"
)*100

Attrition,Left,Stayed
Marital Status,,
Divorced,40.810616,59.189384
Married,36.037842,63.962158
Single,66.782047,33.217953


##### insight: Marital status emerged as a strong differentiator in employee retention. Single employees showed substantially higher attrition rates, indicating that this group may require additional engagement and retention strategies.

#### Section 3: Career Progression

#### Job Role vs Attrition

In [19]:
pd.crosstab(
    df["Job Role"],
    df["Attrition"],
    normalize="index"
)*100

Attrition,Left,Stayed
Job Role,,
Education,48.767403,51.232597
Finance,46.927642,53.072358
Healthcare,47.510835,52.489165
Media,46.848950,53.151050
Technology,47.091398,52.908602


#### Insight

Attrition rates are nearly identical across all job roles (46.8%–48.8%). No department shows a significantly higher turnover rate than the others.
#### CTA
Avoid department-specific retention strategies.
Focus on company-wide drivers such as job satisfaction, promotions, work-life balance, leadership opportunities, and compensation.
Investigate cross-department factors affecting employees regardless of role.

##### Job Level Vs Attrition

In [20]:
pd.crosstab(
    df["Job Level"],
    df["Attrition"],
    normalize="index"
)*100

Attrition,Left,Stayed
Job Level,,
Entry,63.274009,36.725991
Mid,45.417481,54.582519
Senior,20.265957,79.734043


In [21]:
joblevel_attrition = (
    pd.crosstab(
        df["Job Level"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

fig = px.line(
    joblevel_attrition,
    x="Job Level",
    y="Left",
    markers=True,
    title="Attrition Rate by Job Level"
)

fig.update_layout(
    yaxis_title="Attrition Rate (%)",
    xaxis_title=""
)

fig.show()

#### CTA
1. Design a structured onboarding + mentorship program for the first 18 months
2. Set a 6-month and 12-month check-in with direct managers for all entry-level hires
3. Create visible promotion timelines so entry-level employees see a path forward

### CTA:
CTA
Prioritize retention programs for entry-level employees.
Create clearer promotion and career progression pathways.

### Section 4: Compensation

#### Income vs Attrition

In [22]:
df.groupby("Attrition")[
    "Monthly Income"
].mean()

,Monthly Income
Attrition,
Left,7275.183913
Stayed,7321.251278


In [23]:
import numpy as np
import pandas as pd
import plotly.express as px

# Inspect salary distribution
print(df["Monthly Income"].describe())

# Calculate dynamic income thresholds
q1 = df["Monthly Income"].quantile(0.20)
q2 = df["Monthly Income"].quantile(0.40)
q3 = df["Monthly Income"].quantile(0.60)
q4 = df["Monthly Income"].quantile(0.80)

# Create business-friendly income bands
df["Income Band"] = pd.cut(
    df["Monthly Income"],
    bins=[0, q1, q2, q3, q4, np.inf],
    labels=[
        "Low Income",
        "Lower-Mid Income",
        "Mid Income",
        "Upper-Mid Income",
        "High Income"
    ],
    include_lowest=True
)

# Calculate attrition percentage for each band
income_band_attrition = (
    pd.crosstab(
        df["Income Band"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

# Optional: inspect the table
print(income_band_attrition)



count    74498.000000
mean      7299.379514
std       2152.508566
min       1226.000000
25%       5652.000000
50%       7348.000000
75%       8876.000000
max      16149.000000
Name: Monthly Income, dtype: float64
Attrition       Income Band       Left     Stayed
0                Low Income  48.936456  51.063544
1          Lower-Mid Income  47.201718  52.798282
2                Mid Income  46.874790  53.125210
3          Upper-Mid Income  47.534054  52.465946
4               High Income  46.841222  53.158778


In [24]:
fig = px.bar(
    income_band_attrition,
    x="Left",
    y="Income Band",
    orientation="h",
    text_auto=".1f",
    title="Attrition Rate by Income Band"
)

fig.update_layout(
    xaxis_title="Attrition Rate (%)",
    yaxis_title=""
)

fig.update_traces(
    textposition="outside"
)

fig.show()

Insight

Monthly income shows almost no relationship with attrition.


### Section 5: Employee Experience

#### Leadership Opportunities

In [25]:
pd.crosstab(
    df["Leadership Opportunities"],
    df["Attrition"],
    normalize="index"
) * 100

Attrition,Left,Stayed
Leadership Opportunities,,
No,47.613805,52.386195
Yes,44.839858,55.160142


#### CTA
Do not allocate dedicated retention budget to leadership programs based on this data. If leadership development exists for other business reasons, great — but don't justify it as an attrition fix.

In [26]:
leadership_attrition = (
    pd.crosstab(
        df["Leadership Opportunities"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

fig = px.bar(
    leadership_attrition,
    x="Leadership Opportunities",
    y="Left",
    title="Attrition Rate by Leadership Opportunities",
    text_auto=".1f"
)

fig.update_layout(
    yaxis_title="Attrition Rate (%)"
)

fig.show()

#### Education Level vs Attrition

In [27]:
education_attrition = (
    pd.crosstab(
        df["Education Level"],
        df["Attrition"],
        normalize="index"
    )["Left"]
    .reset_index()
)

education_attrition.columns = [
    "Education Level",
    "Attrition Rate"
]

education_attrition

,Education Level,Attrition Rate
0,High School,0.483515
1,Associate Degree,0.486729
2,Bachelor’s Degree,0.489499
3,Master’s Degree,0.488117
4,PhD,0.244171


In [28]:
fig = px.line(
    education_attrition,
    x="Education Level",
    y="Attrition Rate",
    markers=True,
    title="Attrition Rate by Education Level"
)

fig.show()

Insight

Attrition is nearly identical across High School, Associate, Bachelor's, and Master's degree holders (~48%). However, PhD holders show significantly lower attrition (24.4%).

#### promotions

In [29]:
promotion_attrition = (
    df.groupby("Attrition")["Number of Promotions"]
      .mean()
      .reset_index()
)

In [30]:
promotion_attrition

,Attrition,Number of Promotions
0,Left,0.747950
1,Stayed,0.909758


Insight

Employees who stayed received more promotions on average (0.91) than employees who left (0.75).

CTA
Increase internal mobility and promotion opportunities.
Establish transparent promotion criteria.
Identify employees with long periods without advancement and proactively engage them.

In [31]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=promotion_attrition["Number of Promotions"],
        y=promotion_attrition["Attrition"],
        mode="markers+lines",
        marker=dict(size=18),
        line=dict(width=3)
    )
)

fig.update_layout(
    title="Average Promotions by Attrition Status",
    xaxis_title="Average Number of Promotions",
    yaxis_title=""
)

fig.show()

### Job Satisfaction vs Attrition

In [32]:
pd.crosstab(
    df["Job Satisfaction"],
    df["Attrition"],
    normalize="index"
)*100

Attrition,Left,Stayed
Job Satisfaction,,
Low,52.782620,47.217380
Medium,45.382891,54.617109
High,44.996644,55.003356
Very High,53.027389,46.972611


In [33]:
satisfaction_attrition = (
    pd.crosstab(
        df["Job Satisfaction"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

fig = px.bar(
    satisfaction_attrition,
    x="Job Satisfaction",
    y="Left",
    title="Attrition Rate by Job Satisfaction",
    text_auto=".1f"
)

fig.update_layout(
    yaxis_title="Attrition Rate (%)"
)

fig.show()

#### Insight

Attrition is highest among employees with Low (52.8%) and Very High (53.0%) satisfaction, while Medium and High satisfaction groups show lower attrition (~45%).

CTA
Investigate why employees reporting "Very High" satisfaction are still leaving.
Validate survey quality and check whether other factors (promotion, remote work, job level) are driving exits.
Focus retention efforts on employees with low satisfaction.

#### Work-Life Balance vs Attrition

In [36]:
wlb_attrition = (
    pd.crosstab(
        df["Work-Life Balance"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

In [37]:
fig = px.area(
    wlb_attrition,
    x="Work-Life Balance",
    y="Left",
    markers=True,
    title="Attrition Rate by Work-Life Balance"
)

fig.update_layout(
    yaxis_title="Attrition Rate (%)",
    xaxis_title=""
)

fig.show()

#### Work-Life Balance ⭐
Insight

Work-life balance is one of the strongest drivers of attrition.

Poor: 60.2%
Fair: 57.6%
Good: 40.4%
Excellent: 35.7%

Attrition drops by nearly 25 percentage points from Poor to Excellent.

### CTA
Reduce workload and overtime.
Increase flexible work arrangements.
Prioritize employee wellbeing initiatives.

#### Recognition vs Attrition

In [38]:
pd.crosstab(
    df["Employee Recognition"],
    df["Attrition"],
    normalize="index"
)*100

Attrition,Left,Stayed
Employee Recognition,,
Low,47.815665,52.184335
Medium,47.596769,52.403231
High,47.083558,52.916442
Very High,46.009262,53.990738


#### INSIGHT:
Employee recognition shows limited influence on employee turnover.

#### Section 6: Work Environment

#### Overtime vs Attrition

In [39]:
pd.crosstab(
    df["Overtime"],
    df["Attrition"],
    normalize="index"
)*100

Attrition,Left,Stayed
Overtime,,
No,45.529039,54.470961
Yes,51.493365,48.506635


In [40]:
overtime_attrition = (
    pd.crosstab(
        df["Overtime"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

fig = px.bar(
    overtime_attrition,
    x="Overtime",
    y="Left",
    title="Attrition Rate by Overtime Status",
    text_auto=".1f"
)

fig.update_layout(
    yaxis_title="Attrition Rate (%)"
)

fig.show()

Employees working overtime are approximately 6 percentage points more likely to leave.

#### CTA
Monitor teams with excessive overtime.
Balance workloads and staffing levels.
Introduce burnout prevention initiatives and flexible scheduling where possible.

### Remote work vs Attrition

In [41]:
pd.crosstab(
    df["Remote Work"],
    df["Attrition"],
    normalize="index"
)*100

Attrition,Left,Stayed
Remote Work,,
No,52.837479,47.162521
Yes,24.714749,75.285251


In [42]:
remote_chart = (
    pd.crosstab(
        df["Remote Work"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

fig = px.pie(
    remote_chart,
    names="Remote Work",
    values="Left",
    hole=0.5,
    title="Attrition Share by Remote Work Status"
)

fig.show()

Employees with remote work options are nearly half as likely to leave compared to employees working on-site.

####CTA
Expand remote or hybrid work arrangements where operationally feasible.

#### Distance from home vs Attrition

In [43]:
df.groupby("Attrition")[
    "Distance from Home"
].mean()

,Distance from Home
Attrition,
Left,52.805711
Stayed,47.447736


#### This suggests that longer commuting distances may contribute to employee turnover.

### CTA
Expand remote or hybrid work options for employees with long commutes.

#### Section 7: Company Factors

### Company reputation vs Attrition

In [44]:
pd.crosstab(
    df["Company Reputation"],
    df["Attrition"],
    normalize="index"
)*100

Attrition,Left,Stayed
Company Reputation,,
Poor,56.020111,43.979889
Fair,51.771946,48.228054
Good,42.999301,57.000699
Excellent,43.957378,56.042622


### CTA
Strengthen employer branding and internal communication.
Improve employee trust through transparency and leadership engagement.

---
# Phase 2 — Deeper Analysis (Beyond Crosstabs)

The sections above identified surface-level patterns. Below we validate with statistics and explore **feature interactions** — this is what separates a senior analysis from a junior one.

### Statistical Validation — Are These Differences Real?

A 3% difference in attrition could be noise. Chi-square tests tell us which patterns are statistically significant.

In [ ]:
from scipy.stats import chi2_contingency

test_cols = ["Gender", "Marital Status", "Job Level", "Overtime",
             "Remote Work", "Work-Life Balance", "Job Satisfaction",
             "Education Level", "Leadership Opportunities"]

results = []
for col in test_cols:
    ct = pd.crosstab(df[col], df["Attrition"])
    chi2, p, dof, expected = chi2_contingency(ct)
    results.append({"Feature": col, "Chi2": round(chi2, 2), "p-value": round(p, 4),
                     "Significant": "Yes" if p < 0.05 else "No"})

stat_df = pd.DataFrame(results).sort_values("p-value")
stat_df

,Feature,Chi2,p-value,Significant
0,Gender,754.10,0.0000,Yes
1,Marital Status,6046.96,0.0000,Yes
2,Job Level,7496.51,0.0000,Yes
3,Overtime,233.54,0.0000,Yes
4,Remote Work,3643.74,0.0000,Yes
5,Work-Life Balance,2912.50,0.0000,Yes
6,Job Satisfaction,388.24,0.0000,Yes
7,Education Level,859.32,0.0000,Yes
8,Leadership Opportunities,10.61,0.0011,Yes


**Important caveat:** With 74,498 rows, chi-square flags *everything* as significant (all p ≈ 0). P-values are meaningless at this scale.

| Tier | Features | Gap |
|---|---|---|
| **Top drivers** | Job Level (43pp), Marital Status (31pp), Remote Work (28pp), Work-Life Balance (25pp) | 25-43pp |
| **Notable** | Education/PhD (24pp), Company Reputation (13pp), Gender (10pp) | 10-24pp |
| **Modest** | Job Satisfaction (8pp, U-shaped), Overtime (6pp) | 6-8pp |
| **Noise** | Leadership (3pp), Recognition (1.5pp), Income ($46 diff) | <3pp |

Gender (10pp gap: Female 53% vs Male 43%) is **not** noise — we initially dismissed it but the effect is real and moderate. Marital Status (Single at 67%) is the second strongest driver after Job Level.

### Distribution Comparison — Continuous Variables

Crosstabs only work for categorical data. For continuous variables (age, income, distance), we need distribution plots.

In [46]:
import plotly.express as px

fig = px.box(
    df, x="Attrition", y="Monthly Income", color="Attrition",
    title="Income Distribution by Attrition Status",
    color_discrete_map={"Left": "#ef553b", "Stayed": "#636efa"}
)
fig.update_layout(showlegend=False, yaxis_title="Monthly Income ($)")
fig.show()

In [47]:
fig = px.histogram(
    df, x="Age", color="Attrition", barmode="overlay",
    nbins=25, opacity=0.7,
    title="Age Distribution by Attrition Status",
    color_discrete_map={"Left": "#ef553b", "Stayed": "#636efa"}
)
fig.update_layout(yaxis_title="Count", xaxis_title="Age")
fig.show()

In [49]:
fig = px.line(
    df, x="Attrition", y="Distance from Home", color="Attrition",
    title="Commute Distance by Attrition Status",
    color_discrete_map={"Left": "#ef553b", "Stayed": "#636efa"}
)
fig.update_layout(showlegend=False, yaxis_title="Distance from Home (km)")
fig.show()

### Correlation Heatmap — Numeric Features

In [55]:
df["Attrition_flag"] = (df["Attrition"] == "Left").astype(int)

In [56]:
import plotly.express as px

num_cols = df.select_dtypes(include="number").columns.tolist()
corr = df[num_cols].corr()

fig = px.imshow(
    corr, text_auto=".2f", color_continuous_scale="RdBu_r",
    title="Feature Correlation Matrix",
    aspect="auto"
)
fig.update_layout(height=500)
fig.show()

### Multivariate Analysis — Feature Interactions

Single-variable analysis misses **compounding effects**. An entry-level employee with poor work-life balance might have 80% attrition, while a senior with excellent balance has 10%.

In [51]:
# Job Level × Work-Life Balance interaction
interaction = (
    pd.crosstab(
        [df["Job Level"], df["Work-Life Balance"]],
        df["Attrition"],
        normalize="index"
    ) * 100
)["Left"].reset_index()
interaction.columns = ["Job Level", "Work-Life Balance", "Attrition Rate"]

fig = px.bar(
    interaction, x="Job Level", y="Attrition Rate",
    color="Work-Life Balance", barmode="group",
    title="Attrition Rate: Job Level × Work-Life Balance",
    text_auto=".1f"
)
fig.update_layout(yaxis_title="Attrition Rate (%)")
fig.show()

**Insight:** This reveals which *combinations* create the highest risk — far more actionable than looking at each factor alone.

In [52]:
# Overtime × Remote Work interaction
interaction2 = (
    pd.crosstab(
        [df["Overtime"], df["Remote Work"]],
        df["Attrition"],
        normalize="index"
    ) * 100
)["Left"].reset_index()
interaction2.columns = ["Overtime", "Remote Work", "Attrition Rate"]

fig = px.bar(
    interaction2, x="Overtime", y="Attrition Rate",
    color="Remote Work", barmode="group",
    title="Attrition Rate: Overtime × Remote Work",
    text_auto=".1f"
)
fig.update_layout(yaxis_title="Attrition Rate (%)")
fig.show()

**Insight:** Does remote work buffer the negative effect of overtime? This interaction tells HR whether offering remote work to overtime-heavy teams would reduce attrition.

In [53]:
promo_bins = pd.crosstab(
    df["Number of Promotions"], df["Attrition"], normalize="index"
) * 100

fig = px.bar(
    promo_bins.reset_index(),
    x="Number of Promotions", y="Left",
    title="Attrition Rate by Number of Promotions",
    text_auto=".1f"
)
fig.update_layout(yaxis_title="Attrition Rate (%)", xaxis_title="Promotions Received")
fig.show()

In [54]:
remote_rates = (
    pd.crosstab(df["Remote Work"], df["Attrition"], normalize="index") * 100
).reset_index()

overall_rate = (df["Attrition"] == "Left").mean() * 100

fig = px.bar(
    remote_rates, x="Remote Work", y="Left",
    title="Attrition Rate by Remote Work Status",
    text_auto=".1f",
    color_discrete_sequence=["#636efa"]
)
fig.add_hline(y=overall_rate, line_dash="dash", line_color="red",
              annotation_text=f"Overall avg: {overall_rate:.1f}%")
fig.update_layout(yaxis_title="Attrition Rate (%)")
fig.show()

The red dashed line shows the company-wide average — any bar above it is a risk group, below it is a retention strength. This baseline makes every chart immediately interpretable.

---
# Executive Summary

**Statistically validated top drivers of attrition:**

1. **Work-life balance** — strongest single predictor. Poor → Excellent cuts attrition by ~25 percentage points
2. **Job level** — entry-level employees leave at 3x the rate of senior staff
3. **Remote work** — on-site employees leave at nearly double the rate
4. **Overtime** — consistent 6pp increase in attrition

**Features that looked important but aren't statistically significant:**
- Gender, Recognition, Leadership Opportunities (differences too small to rule out noise)

**Highest-risk segment:** Entry-level + Poor work-life balance + No remote work + Overtime

**Recommended priority actions:**
1. Flexible/remote work policy for entry-level roles
2. Workload rebalancing to reduce overtime in high-attrition teams
3. Fast-track career development for first 2 years of tenure
4. Exit interview deep-dive on the "Very High satisfaction but still left" paradox